# Stage 2: Exploratory Data Analysis & Data Quality Audit
**Project:** ML-NIDS with Random Forest and SHAP Explainable AI  
**Objective:** Evaluate class distributions, measure statistical profiles across Benign vs. Attack traffic, audit information leakage risks (IP addresses and ports), and identify feature correlations.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Workspace path
project_root = Path("..").resolve()
sys.path.insert(0, str(project_root))

from src.data.loaders import load_flow_dataset
from src.data.eda import run_full_eda

## 1. Execute Complete EDA & Quality Audit Pipeline
This runs the quality audit, calculates leakage metrics, and generates figures in `artifacts/figures/`.

In [ ]:
eda_summary = run_full_eda(project_root)
import json
print(json.dumps(eda_summary, indent=2))

## 2. Load Biflow Dataset for Interactive Analysis

In [ ]:
biflow_df = load_flow_dataset('biflow', base_dir=project_root)
print(f"Total biflow records loaded: {len(biflow_df):,}")
biflow_df.head(5)

## 3. Class Balance Across Attack Scenarios

In [ ]:
ct = pd.crosstab(biflow_df['scenario'], biflow_df['is_attack'], margins=True)
ct.columns = ['Benign (0)', 'Attack (1)', 'Total']
ct['Attack %'] = (ct['Attack (1)'] / ct['Total'] * 100).round(2)
display(ct)

## 4. Key Flow Statistics: Benign vs. Attack
Compare mean packet lengths, inter-arrival times, packet counts, and TCP flags.

In [ ]:
cols_to_compare = [
    'fwd_num_pkts', 'bwd_num_pkts', 'fwd_mean_iat',
    'fwd_mean_pkt_len', 'fwd_num_bytes', 'fwd_num_psh_flags', 'bwd_num_rst_flags'
]
display(biflow_df.groupby('is_attack')[cols_to_compare].mean().T)

## 5. Critical Finding: Predictive Leakage Audit on IP Addresses
Notice that in the testbed network, only two IP addresses initiate attacks (`192.168.1.7` and `192.168.2.5`), while all normal traffic is initiated by internal hosts. Including `ip_src` would cause severe model overfitting and memorize host identities rather than attack behavior!

In [ ]:
norm_ips = set(biflow_df[biflow_df['is_attack'] == 0]['ip_src'].unique())
atk_ips = set(biflow_df[biflow_df['is_attack'] == 1]['ip_src'].unique())
print(f"Normal Source IPs ({len(norm_ips)}): {norm_ips}")
print(f"Attack Source IPs ({len(atk_ips)}): {atk_ips}")
print(f"Overlap: {norm_ips.intersection(atk_ips)}")